# Mumbai Urban Heat Island: Physics-Informed Neural Network (PINN) & Equity Optimization
### B.Tech CSE (AIML) - FF-180 Group 16
**Team:** Prajwal Kadam, Atharva Kangralkar  
**Guide:** Mrs. Grishma Bobhate

This interactive notebook demonstrates the complete physics-informed pipeline:
1. Geospatial feature ingestion & coastal distance transforms
2. Random Forest and XGBoost benchmarks
3. Staged PINN with wind-driven sea-breeze advection & parameter discovery
4. SHAP explainability
5. What-if urban greening and cool-roof simulation
6. Equity-aware NSGA-II multi-objective optimization

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## 1. Load Data & Inspect Mumbai Features

In [ ]:
from scripts.data_loader import load_mumbai_data
data = load_mumbai_data(holdout_zone=4)
df_land = data["df_all"]
print(f"Total Land Pixels: {len(df_land):,}")
df_land[["lst_c", "ndvi", "ndbi", "dist_to_sea_km", "building_density", "building_height"]].describe()

## 2. Review Model Benchmarks (RF vs. XGBoost vs. PINN Stages)

In [ ]:
import json
with open(PROJECT_ROOT / "results" / "metrics" / "baseline_metrics.json") as f:
    baseline_metrics = json.load(f)
with open(PROJECT_ROOT / "results" / "metrics" / "pinn_ablation_metrics.json") as f:
    pinn_metrics = json.load(f)

summary = [
    {"Model": "Random Forest", "Val RMSE (°C)": baseline_metrics["random_forest"]["validation"]["rmse"], "Holdout RMSE (°C)": baseline_metrics["random_forest"]["spatial_holdout"]["rmse"]},
    {"Model": "XGBoost", "Val RMSE (°C)": baseline_metrics["xgboost"]["validation"]["rmse"], "Holdout RMSE (°C)": baseline_metrics["xgboost"]["spatial_holdout"]["rmse"]},
    {"Model": "PINN Stage 1 (Pure Data)", "Val RMSE (°C)": pinn_metrics["stage1"]["val_rmse"], "Holdout RMSE (°C)": pinn_metrics["stage1"]["test_rmse"]},
    {"Model": "PINN Stage 2 (+Diffusion)", "Val RMSE (°C)": pinn_metrics["stage2"]["val_rmse"], "Holdout RMSE (°C)": pinn_metrics["stage2"]["test_rmse"]},
    {"Model": "PINN Stage 3 (+Sea-Breeze)", "Val RMSE (°C)": pinn_metrics["stage3"]["val_rmse"], "Holdout RMSE (°C)": pinn_metrics["stage3"]["test_rmse"]},
]
pd.DataFrame(summary)

## 3. Equity-Aware Optimization Policy Recommendations

In [ ]:
recs_df = pd.read_csv(PROJECT_ROOT / "results" / "metrics" / "nsga2_recommendations.csv")
recs_df

## 4. Visualizations Gallery (All 12 Spec Figures)

In [ ]:
from IPython.display import Image, display
fig_dir = PROJECT_ROOT / "results" / "figures"

figs_to_show = [
    ("Figure 1: Observed LST Map", fig_dir / "fig1_mumbai_lst_map.png"),
    ("Figure 8: PINN Ablation Bar Chart", fig_dir / "fig8_pinn_ablation_barchart.png"),
    ("Figure 10: NSGA-II Pareto Front", fig_dir / "fig10_nsga2_pareto_front.png"),
    ("Figure 12: Ward Recommendations Map", fig_dir / "fig12_ward_recommendation_map.png"),
]

for title, p in figs_to_show:
    if p.exists():
        print(title)
        display(Image(filename=str(p)))